# When the Sample Table and the Spectra Live in Different Files
### 2.7 — Joining Measurements and Metadata

The instrument exports one file: sample IDs and their readings. The sample
metadata — operator, batch, true concentration — lives in a different file, from
the LIMS or your own worklist. Before you can analyze anything you have to put them
back together on the shared key, the sample ID.

A **join** does exactly that. It's also the moment hidden problems surface: a
sample with no metadata, a duplicate ID that quietly multiplies your rows, two
files that don't line up.

> **One idea to hold onto:** a join reunites measurements and metadata on the
> sample ID — and it's where mismatches (missing metadata, duplicate IDs) become
> visible. Check the row count before and after.

**By the end of this notebook you will be able to:**

1. Merge a measurements table with a metadata table on `sample_id`.
2. Choose an inner vs left join and see missing metadata appear.
3. Catch duplicate-ID row multiplication and validate a merge.

## 1. Two tables from two files

The measurements (from the instrument) and the metadata (from the LIMS). Note the
deliberate mismatch: `S-05` was measured but has no metadata, and `S-09` has
metadata but was never measured.

In [ ]:
import pandas as pd

measurements = pd.DataFrame({
    "sample_id":    ["S-01", "S-02", "S-03", "S-04", "S-05"],
    "absorbance_AU": [0.31, 0.49, 0.13, 0.52, 0.28],
})

metadata = pd.DataFrame({
    "sample_id":         ["S-01", "S-02", "S-03", "S-04", "S-09"],
    "batch":             ["A", "A", "B", "B", "C"],
    "concentration_mgL": [5.0, 8.0, 2.0, 8.1, 3.3],
})

print("measurements:", measurements.shape, "| metadata:", metadata.shape)

## 2. Inner join: only samples in both files

`pd.merge(..., how="inner")` keeps only sample IDs present in *both* tables. It's the
safe default, but notice it silently drops the unmatched samples.

In [ ]:
inner = pd.merge(measurements, metadata, on="sample_id", how="inner")
print("inner join shape:", inner.shape, "(S-05 and S-09 dropped -- only matches kept)")
inner

## 3. Left join: keep every measurement, see missing metadata

A **left** join keeps all rows from the left table (the measurements) and fills
metadata with `NaN` where there's no match. `indicator=True` adds a column showing
where each row came from — so a missing match is visible, not silent.

In [ ]:
left = pd.merge(measurements, metadata, on="sample_id", how="left", indicator=True)
print(left)
print("\nrows with missing metadata:")
print(left[left["_merge"] == "left_only"][["sample_id", "batch"]])

S-05 kept its reading but shows `NaN` metadata and `left_only` — exactly the kind of
gap you want to *see* (and chase down) rather than have quietly dropped.

## 4. Duplicate IDs multiply rows

Here's the join error that corrupts results quietly. If the metadata has a
**duplicate** sample ID, every matching measurement row is duplicated — your sample
count silently grows.

In [ ]:
# Metadata with S-02 listed twice (a data-entry slip).
metadata_dup = pd.concat([metadata, pd.DataFrame({
    "sample_id": ["S-02"], "batch": ["A"], "concentration_mgL": [8.4],
})], ignore_index=True)

bad = pd.merge(measurements, metadata_dup, on="sample_id", how="inner")
print("measurements rows:", len(measurements))
print("after merge with duplicated metadata:", len(bad), "<- S-02 doubled!")
print("\nduplicate keys in metadata:",
      metadata_dup.loc[metadata_dup["sample_id"].duplicated(keep=False), "sample_id"].tolist())

## 5. Validate the merge

You don't have to catch this by eye. `validate="m:1"` tells pandas "many measurements
map to one metadata row" — and it raises an error if the right table's key isn't
unique, turning a silent corruption into a loud failure.

In [ ]:
try:
    pd.merge(measurements, metadata_dup, on="sample_id", how="inner", validate="m:1")
    print("merge passed validation")
except Exception as err:
    print("merge REJECTED (good):", type(err).__name__)
    print(" ", err)

## Key Takeaways

- A **merge** reunites measurements and metadata on a shared key (`sample_id`).
- **Inner** keeps only matches; **left** keeps all measurements and exposes missing
  metadata as `NaN` (use `indicator=True` to see the source).
- **Duplicate keys multiply rows** — a silent way to corrupt a sample count.
- `validate="m:1"` turns a duplicate-key merge into an error instead of bad data.

## Practical Checklist

- [ ] Decide which table must be kept whole, and pick inner vs left accordingly.
- [ ] Compare row counts before and after every merge.
- [ ] Check both keys for duplicates before joining.
- [ ] Use `validate=` to assert the relationship you expect (`"1:1"`, `"m:1"`).

## Common Mistakes

- Assuming a merge is lossless — an inner join silently drops unmatched samples.
- Merging on a key with duplicates and not noticing the row count grew.
- Ignoring rows with missing metadata instead of investigating them.

## Next Lesson

**2.8 — Reshaping Data for Analysis.** Joined and clean, the data still isn't in the
shape a model wants; next we reshape it to one row per spectrum.